# Colab B — Download shard 1/3

Dополнительный воркер к **Colab A** (discover).
State локально `/content/` → Drive синк каждые 2 мин. Видео → HF напрямую.

> **Runbook:** `Fetcher/docs/COLAB_20K_RUN_RU.md`

## 0. CONFIG

In [ ]:
CONFIG = {
    "repo_url":            "https://github.com/lebedev-ilia/TrendFlowML.git",
    "fetcher":             "/content/TrendFlowML/Fetcher",
    "drive_secrets":       "/content/drive/MyDrive/trendflow_secrets",
    "output_dir":          "/content/dataset_runs/20k-worker-b",  # локально
    "drive_backup_dir":    "/content/drive/MyDrive/dataset_runs/20k-worker-b",
    "hf_repo_prefix":      "Ilialebedev",
    "campaign_profile":    "20k",
    "worker_id":           "colab-b",
    "worker_shard_index":  1,
    "worker_shard_count":  3,
    "parallel_colab_count": 3,
    "download_pause_after_success_seconds": 5,
    "download_pause_after_bot_seconds":     120,
    "hf_commit_min_interval_seconds":       95,
    "hf_parallel_colab_count":             3,
    "sync_interval_sec":   120,
    "metrics_workers":     9096,
}

import json
from pathlib import Path

OUTPUT = Path(CONFIG["output_dir"])
OUTPUT.mkdir(parents=True, exist_ok=True)

OVERRIDES = {k: CONFIG[k] for k in [
    "worker_id", "worker_shard_index", "worker_shard_count",
    "hf_parallel_colab_count", "hf_commit_min_interval_seconds",
    "download_pause_after_success_seconds", "download_pause_after_bot_seconds",
] if k in CONFIG}
OVERRIDES_PATH = OUTPUT / "config_overrides_b.json"
OVERRIDES_PATH.write_text(json.dumps(OVERRIDES, indent=2), encoding="utf-8")

print("worker_id :", CONFIG["worker_id"])
print("shard     :", CONFIG["worker_shard_index"], "/", CONFIG["worker_shard_count"])
print("output_dir:", OUTPUT)
print("backup_dir:", CONFIG["drive_backup_dir"])


## 1. Drive + git + pip

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess
from pathlib import Path

dest = Path("/content/TrendFlowML")
if (dest / ".git").exists():
    subprocess.run(["git", "-C", str(dest), "pull"], check=False)
else:
    subprocess.run(["git", "clone", CONFIG["repo_url"], str(dest)], check=True)

subprocess.run(["pip", "install", "-q", "-r",
    str(Path(CONFIG["fetcher"]) / "requirements.txt")], check=True)
subprocess.run(["pip", "install", "-q", "-U",
    "huggingface_hub", "yt-dlp", "pytubefix>=9.5.0",
    "nodejs-wheel-binaries", "google-api-python-client"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "nodejs"], check=False)
print("install done")


## 2. HF_TOKEN + cookies

`HF_TOKEN` → добавьте в **Colab Secrets** (🔑).
YouTube ключи здесь не нужны — discover только на Colab A.

In [ ]:
import os, shutil
from pathlib import Path

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN").strip()
except Exception:
    pass
# os.environ["HF_TOKEN"] = "hf_..."

if not os.environ.get("HF_TOKEN", "").startswith("hf_"):
    raise RuntimeError("Добавьте HF_TOKEN в Colab Secrets или впишите вручную")

Path(CONFIG["output_dir"]).joinpath(".hf_token").write_text(
    os.environ["HF_TOKEN"].strip() + "\n", encoding="utf-8")
print("HF_TOKEN OK")

secrets = Path(CONFIG["drive_secrets"])
cookie_dst = Path(CONFIG["fetcher"]) / "fetcher/dataset_collector/cookies"
cookie_dst.mkdir(parents=True, exist_ok=True)
cookie_src = secrets / "cookies"
if cookie_src.is_dir():
    for f in cookie_src.glob("*.txt"):
        shutil.copy2(f, cookie_dst / f.name)
    print(f"Cookies: {len(list(cookie_dst.glob('*.txt')))} файлов")
else:
    print("WARN: cookies не найдены — скачивание может упасть на bot-detection")


## 3. Restore state с Drive (resume после обрыва)

In [ ]:
import subprocess
from pathlib import Path

backup = Path(CONFIG["drive_backup_dir"])
local  = Path(CONFIG["output_dir"])
local.mkdir(parents=True, exist_ok=True)

restored = []
for d in ["state", "shards", "downloads"]:
    src = backup / d
    if src.is_dir():
        dst = local / d
        dst.mkdir(parents=True, exist_ok=True)
        subprocess.run(["rsync", "-a", "--ignore-existing",
                        str(src) + "/", str(dst) + "/"], check=False)
        restored.append(d)

print("Restored from Drive:", restored if restored else "ничего (первый запуск)")


## 4. Auto-sync state → Drive (каждые 2 мин)

⚠️ Запустите **один раз** — работает в фоне весь сеанс.

In [ ]:
import threading, subprocess, time
from pathlib import Path

_sync_stop = False

def _do_sync():
    local  = Path(CONFIG["output_dir"])
    backup = Path(CONFIG["drive_backup_dir"])
    backup.mkdir(parents=True, exist_ok=True)
    synced = []
    for d in ["state", "shards", "downloads"]:
        src = local / d
        if not src.is_dir():
            continue
        dst = backup / d
        dst.mkdir(parents=True, exist_ok=True)
        r = subprocess.run(
            ["rsync", "-a",
             "--exclude=*.mp4", "--exclude=*.video.tmp", "--exclude=*.audio.tmp",
             str(src) + "/", str(dst) + "/"],
            capture_output=True
        )
        if r.returncode == 0:
            synced.append(d)
    return synced

def _sync_loop():
    interval = CONFIG["sync_interval_sec"]
    while not _sync_stop:
        time.sleep(interval)
        try:
            synced = _do_sync()
            print(f"[sync {time.strftime('%H:%M:%S')}] → Drive: {synced}")
        except Exception as e:
            print(f"[sync ERROR] {e}")

_t = threading.Thread(target=_sync_loop, daemon=True, name="drive-sync")
_t.start()
print(f"Auto-sync запущен каждые {CONFIG['sync_interval_sec']}s → {CONFIG['drive_backup_dir']}")


## 5. Workers-download shard 1/3

Запускается после того как Colab A поднял discover + shards upload.
Если очередь пуста в начале — норм, через 2-3 мин появятся задачи.

In [ ]:
import subprocess, os
from pathlib import Path

cmd = [
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile",    CONFIG["campaign_profile"],
    "--role",                "workers-download",
    "--output-dir",          CONFIG["output_dir"],
    "--hf-repo-prefix",      CONFIG["hf_repo_prefix"],
    "--worker-id",           CONFIG["worker_id"],
    "--worker-shard-index",  str(CONFIG["worker_shard_index"]),
    "--worker-shard-count",  str(CONFIG["worker_shard_count"]),
    "--parallel-colab-count", str(CONFIG["parallel_colab_count"]),
    "--cookie-files-dir",    "fetcher/dataset_collector/cookies",
    "--metrics-port",        str(CONFIG["metrics_workers"]),
    "--config-overrides-json", str(OVERRIDES_PATH),
]
log = Path("/content/workers_b_log.txt")
with open(log, "w") as fh:
    p = subprocess.Popen(cmd, cwd=CONFIG["fetcher"],
                         stdout=fh, stderr=subprocess.STDOUT,
                         env=os.environ.copy())
print(f"workers-download pid={p.pid} shard=1/3  →  !tail -f {log}")
print()
print("Ждём sync метаданных с HF (Colab A). Если очередь пуста — подожди 2-3 мин.")


## 6. Статус

In [ ]:
import subprocess, os
subprocess.run([
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile", CONFIG["campaign_profile"],
    "--role",             "status",
    "--output-dir",       CONFIG["output_dir"],
], cwd=CONFIG["fetcher"], env=os.environ.copy(), check=False)


## 7. Принудительный sync

In [ ]:
import time
synced = _do_sync()
print(f"[{time.strftime('%H:%M:%S')}] Sync → Drive: {synced}")
